# Ejercicios de práctica - Sesión 02

En este notebook vas a practicar lo visto en la sesión: consumo de LLMs mediante API, uso de instrucciones, conversación multi-turno, gestión básica de tokens, streaming, salida estructurada y una primera capa de abstracción con LangChain.

El propósito es que completes los huecos, ejecutes las celdas y compares cómo cambia la forma de trabajar cuando llamas directamente a OpenAI, a Google Gemini o a los modelos a través de LangChain.

## Cómo trabajar este notebook

Completa las celdas marcadas con `TODO`. Puedes cambiar los textos de ejemplo si quieres probar otros casos, pero mantén los nombres de variables principales para que las celdas posteriores sigan funcionando.

Usaremos los mismos modelos de la sesión:

- OpenAI: `gpt-4.1-nano`
- Google Gemini: `gemini-2.5-flash-lite`

No escribas claves de API directamente en el notebook. Usa un archivo `.env` local o variables de entorno.

## 0. Preparación

Si no tienes las dependencias instaladas, descomenta y ejecuta la siguiente celda. Si ya has preparado el entorno con `make setup` o con `requirements.txt`, puedes saltarla.

In [ ]:
# %pip install -q -U openai google-genai langchain langchain-openai langchain-google-genai langgraph python-dotenv tiktoken pydantic

Carga las credenciales desde `.env` o desde variables de entorno. Si falta alguna clave, la celda la pedirá de forma interactiva.

In [ ]:
import getpass
import os

from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google API key: ")

OPENAI_MODEL = "gpt-4.1-nano"
GOOGLE_MODEL = "gemini-2.5-flash-lite"

print("Credenciales cargadas y modelos definidos.")

In [ ]:
from openai import OpenAI
from google import genai
from google.genai import types as genai_types

openai_client = OpenAI()
gemini_client = genai.Client()

## 1. Primera llamada directa a dos proveedores

Empieza con la misma tarea en dos APIs distintas: traducir una frase de inglés a italiano. El objetivo no es que el resultado sea idéntico, sino que veas qué partes se repiten: cliente, modelo, input y lectura de la respuesta.

In [ ]:
def traduce_con_openai(texto: str) -> str:
    """Devuelve una traducción al italiano usando OpenAI Responses API."""
    # TODO: llama a openai_client.responses.create usando OPENAI_MODEL.
    # Pista: response.output_text contiene el texto final.
    raise NotImplementedError("Completa la llamada a OpenAI.")


def traduce_con_gemini(texto: str) -> str:
    """Devuelve una traducción al italiano usando Gemini API."""
    # TODO: llama a gemini_client.models.generate_content usando GOOGLE_MODEL.
    # Pista: response.text contiene el texto final.
    raise NotImplementedError("Completa la llamada a Gemini.")

texto = "Hello, how are you?"

print("OpenAI:", traduce_con_openai(texto))
print("Gemini:", traduce_con_gemini(texto))

## 2. Instrucciones del sistema y roles

Ahora separa la instrucción general del mensaje del usuario. En una aplicación real, esta separación es importante porque permite fijar el comportamiento del asistente sin mezclarlo con cada input.

Crea un asistente que ayude a aprender italiano: debe responder en español, dar la traducción en italiano y añadir una nota breve de uso.

In [ ]:
INSTRUCCIONES_TUTOR_ITALIANO = """
Eres un tutor de italiano para hispanohablantes.
Responde en español.
Cuando traduzcas una frase, incluye la traducción italiana y una nota breve sobre registro o uso.
Sé conciso.
""".strip()

pregunta = "How would I say 'Good morning, nice to meet you' in Italian?"

In [ ]:
def tutor_openai(pregunta: str) -> str:
    # TODO: usa `instructions=INSTRUCCIONES_TUTOR_ITALIANO` y `input=pregunta`.
    raise NotImplementedError("Completa la llamada con instrucciones en OpenAI.")


def tutor_gemini(pregunta: str) -> str:
    # TODO: usa GenerateContentConfig(system_instruction=INSTRUCCIONES_TUTOR_ITALIANO).
    raise NotImplementedError("Completa la llamada con system_instruction en Gemini.")

print("OpenAI:\n", tutor_openai(pregunta))
print("\nGemini:\n", tutor_gemini(pregunta))

## 3. Conversación multi-turno

Un modelo no recuerda por sí solo lo que se habló en una llamada anterior. Practica dos formas de dar continuidad a una conversación: `previous_response_id` en OpenAI y el objeto `Chat` en Gemini.

In [ ]:
primera_pregunta = "How do you say 'See you tomorrow' in Italian?"
segunda_pregunta = "Is that formal or informal?"

In [ ]:
def conversacion_openai() -> tuple[str, str]:
    # TODO: crea una primera respuesta con `instructions=INSTRUCCIONES_TUTOR_ITALIANO`.
    # TODO: crea una segunda respuesta usando `previous_response_id=primera_respuesta.id`.
    # Devuelve una tupla con los textos de ambas respuestas.
    raise NotImplementedError("Completa la conversación multi-turno con OpenAI.")


respuesta_1, respuesta_2 = conversacion_openai()
print("Primer turno:\n", respuesta_1)
print("\nSegundo turno:\n", respuesta_2)


In [ ]:
def conversacion_gemini() -> list[str]:
    # TODO: crea un chat con gemini_client.chats.create(model=GOOGLE_MODEL).
    # TODO: envía `primera_pregunta` y `segunda_pregunta` con chat.send_message(...).
    # Devuelve una lista con los textos de ambas respuestas.
    raise NotImplementedError("Completa la conversación multi-turno con Gemini.")


respuestas = conversacion_gemini()
for i, respuesta in enumerate(respuestas, start=1):
    print(f"Turno {i}:\n{respuesta}\n")


## 4. Tokens, coste y elección de modelo

En esta parte vas a observar dos cosas: los límites declarados por Gemini para el modelo elegido y el uso real de tokens que devuelve una llamada de OpenAI. Después, resume qué modelo elegirías para distintos casos.

In [ ]:
model_info = gemini_client.models.get(model=GOOGLE_MODEL)

print("Gemini model:", GOOGLE_MODEL)
print("Input token limit:", model_info.input_token_limit)
print("Output token limit:", model_info.output_token_limit)

In [ ]:
texto_largo = (
    "Los LLMs consumidos por API permiten crear aplicaciones sin entrenar un modelo desde cero. "
    "A cambio, debemos gestionar coste, latencia, privacidad, contexto y formato de salida. "
) * 80

conteo = gemini_client.models.count_tokens(
    model=GOOGLE_MODEL,
    contents=texto_largo,
)

print("Tokens estimados en Gemini:", conteo.total_tokens)

In [ ]:
response = openai_client.responses.create(
    model=OPENAI_MODEL,
    input="Resume en una frase qué criterio usarías para elegir un modelo de LLM por API.",
)

print(response.output_text)
print("Input tokens:", response.usage.input_tokens)
print("Output tokens:", response.usage.output_tokens)
print("Total tokens:", response.usage.total_tokens)

Completa esta reflexión antes de seguir. Escribe una respuesta breve para cada caso.

**Prototipo barato para testeo:**


**Tarea compleja de análisis o programación:**


**Chat con documentos largos:**


**Aplicación con datos sensibles:**


**Respuesta visible en una interfaz conversacional:**


## 5. Streaming

El streaming no cambia necesariamente la calidad de la respuesta, pero sí cambia la experiencia de usuario: permite mostrar fragmentos a medida que se generan.

Completa uno de los dos ejemplos. Si quieres practicar más, completa ambos.

In [ ]:
# Streaming con OpenAI
# TODO: crea un stream con openai_client.responses.create(..., stream=True).
# Después, recorre los eventos e imprime solo los deltas de texto.
# Pista: en la sesión se usó event.type == "response.output_text.delta".

raise NotImplementedError("Completa el ejemplo de streaming con OpenAI.")


In [ ]:
# Streaming con Gemini
# TODO: crea un stream con gemini_client.models.generate_content_stream(...).
# Después, recorre los chunks e imprime chunk.text.

raise NotImplementedError("Completa el ejemplo de streaming con Gemini.")


## 6. Salida estructurada en JSON

Cuando el resultado de un LLM se integra en software, el texto libre suele ser insuficiente. En esta práctica vas a pedir una salida con campos concretos: traducción principal, registro y alternativas.

In [ ]:
import json

openai_translation_schema = {
    "type": "object",
    "properties": {
        "translation": {"type": "string"},
        "register": {"type": "string", "enum": ["formal", "informal", "neutral"]},
        "alternatives": {
            "type": "array",
            "items": {"type": "string"},
        },
    },
    "required": ["translation", "register", "alternatives"],
    "additionalProperties": False,
}

gemini_translation_schema = {
    "type": "object",
    "properties": {
        "translation": {"type": "string"},
        "register": {"type": "string", "enum": ["formal", "informal", "neutral"]},
        "alternatives": {
            "type": "array",
            "items": {"type": "string"},
        },
    },
    "required": ["translation", "register", "alternatives"],
}


In [ ]:
def json_openai(texto: str) -> dict:
    # TODO: usa `text={"format": {"type": "json_schema", ...}}` como en la sesión.
    # Usa `openai_translation_schema` y devuelve un dict de Python, no un string.
    raise NotImplementedError("Completa la salida JSON con OpenAI.")


json_openai("Good evening, how are you?")


In [ ]:
def json_gemini(texto: str) -> dict:
    # TODO: usa GenerateContentConfig(response_mime_type="application/json", response_schema=gemini_translation_schema).
    # Devuelve un dict de Python, no un string.
    raise NotImplementedError("Completa la salida JSON con Gemini.")


json_gemini("Good evening, how are you?")


## 7. LangChain como capa común

LangChain permite expresar parte del flujo de forma más independiente del proveedor. En este ejercicio usarás un `ChatPromptTemplate` y después pedirás salida estructurada con un modelo de OpenAI o Gemini.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Eres un tutor de idiomas. Responde de forma breve y clara."),
        ("user", "Traduce a {idioma}: {texto}"),
    ]
)

# Puedes cambiar a GOOGLE_MODEL con model_provider="google_genai" si prefieres usar Gemini.
langchain_model = init_chat_model(OPENAI_MODEL, model_provider="openai")

formatted_prompt = prompt.invoke({"idioma": "italiano", "texto": "I am learning to use language models through APIs."})
response = langchain_model.invoke(formatted_prompt)
response.content

In [ ]:
from pydantic import BaseModel, Field

class TranslationResult(BaseModel):
    translation: str = Field(description="Traducción principal")
    register: str = Field(description="Registro: formal, informal o neutral")
    alternatives: list[str] = Field(description="Traducciones alternativas")

# TODO: crea un modelo estructurado con langchain_model.with_structured_output(TranslationResult).
# TODO: invoca el modelo estructurado con una petición de traducción.
raise NotImplementedError("Completa la salida estructurada con LangChain.")


## 8. Mini reto final

Construye una función que permita elegir proveedor sin cambiar el resto del código. La función debe devolver siempre un diccionario con la misma forma, aunque por debajo use OpenAI o Gemini.

Forma esperada:

```python
{
    "provider": "openai" | "gemini",
    "model": "...",
    "translation": "..."
}
```

In [ ]:
def traduce(provider: str, texto: str, idioma: str = "italiano") -> dict:
    provider = provider.lower().strip()

    if provider == "openai":
        # TODO: llama a OpenAI y devuelve el diccionario esperado.
        raise NotImplementedError("Completa la rama de OpenAI.")

    if provider == "gemini":
        # TODO: llama a Gemini y devuelve el diccionario esperado.
        raise NotImplementedError("Completa la rama de Gemini.")

    raise ValueError("provider debe ser 'openai' o 'gemini'")


print(traduce("openai", "Good luck with the course"))
print(traduce("gemini", "Good luck with the course"))

## Cierre

Al terminar, revisa qué partes son iguales entre proveedores y cuáles cambian. Esa comparación es el objetivo principal de la práctica: aprender a usar LLMs vía API sin depender mentalmente de una única plataforma.